# Chapter 8 experiments — Advanced Scheduling on a Colab T4

Companion notebook to [`book/08-advanced-scheduling.md`](../book/08-advanced-scheduling.md):
runnable versions of the chapter's four claims.

| # | Book section | Experiment |
|---|---|---|
| 1 | Chunked Prefill | prefill time grows superlinearly with prompt length |
| 2 | Chunked Prefill | chunking bounds the decode stall — and the TTFT cost |
| 3 | Mixed Batches | padding waste (~33% efficiency) vs real-token batching |
| 4 | CUDA Graphs | launch overhead: eager vs graph replay across shapes |
| 5 | Overlap Scheduling | hide CPU scheduling behind GPU work (+ graphs combo) |
| 6 | CUDA Graphs / torch.compile | fusion + automatic CUDA graphs vs the manual runner |

**Setup:** `Runtime -> Change runtime type -> T4 GPU`. Pure PyTorch, no installs.

**Reading the numbers:** the model is a ~78M-param random-weight MiniGPT
(24 layers, ~13 kernels/layer = **300+ launches per step** — the book's number).
Absolute times are ~100x smaller than a 7B on an H100; the *ratios* are what
reproduce the chapter's claims.

**T4 caveats:** everything runs fp16 (Turing has no bf16/TF32). FlashInfer needs
SM80+ (A100/H100), so the chapter's FlashInfer kernels can't run here —
experiment 3 approximates the mixed batch with plain PyTorch.

In [ ]:
import torch, time, threading
import torch.nn as nn
import torch.nn.functional as F

device = "cuda"
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))  # Tesla T4 (7, 5)

_masks = {}
def causal_mask(T, dev):
    if (T, dev) not in _masks:
        _masks[(T, dev)] = torch.triu(torch.ones(T, T, device=dev, dtype=torch.bool), 1)
    return _masks[(T, dev)]

class Block(nn.Module):
    def __init__(self, d=512, h=8):
        super().__init__()
        self.h, self.dk = h, d // h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3*d, bias=False), nn.Linear(d, d, bias=False)
        self.fc1, self.fc2 = nn.Linear(d, 4*d, bias=False), nn.Linear(4*d, d, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        z = self.ln1(x)
        qkv = self.qkv(z).view(B, T, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q @ k.transpose(-2, -1)) * self.dk ** -0.5
        if T > 1:                                  # decode (T=1) needs no mask
            att = att.masked_fill(causal_mask(T, x.device), float("-inf"))
        y = (F.softmax(att, dim=-1) @ v).transpose(1, 2).reshape(B, T, C)
        x = x + self.proj(y)
        z = self.ln2(x)
        return x + self.fc2(F.gelu(self.fc1(z)))

class MiniGPT(nn.Module):
    def __init__(self, vocab=2048, d=512, n_layers=24, h=8):
        super().__init__()
        self.emb, self.ln_f = nn.Embedding(vocab, d), nn.LayerNorm(d)
        self.blocks = nn.ModuleList(Block(d, h) for _ in range(n_layers))
        self.head = nn.Linear(d, vocab, bias=False)
    def forward(self, ids):                        # ids: (B, T) long
        x = self.emb(ids)
        for b in self.blocks: x = b(x)
        return self.head(self.ln_f(x))

torch.manual_seed(0)
model = MiniGPT().half().to(device).eval()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

def time_ms(fn, warmup=10, iters=30):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(True), torch.cuda.Event(True)
    s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / iters

## 1. Why long prefills hurt

The chapter's claim: prefill is compute-bound and grows ~quadratically with
prompt length, so a 32K prompt blocks every concurrent decode for seconds.
Time grows **superlinearly** — linear layers scale with T, the attention term
with T^2 — and at the largest T, that whole duration is the stall a concurrent
decode token would experience.

In [ ]:
for T in [512, 1024, 2048, 4096]:
    ids = torch.randint(0, 2048, (1, T), device=device)
    print(f"T={T:5d}   prefill {time_ms(lambda: model(ids)):8.2f} ms")

## 2. Chunked prefill

Claim under test: breaking the prefill into 512-token chunks bounds any decode's
stall to one chunk — at the cost of the long prompt's own TTFT. The book's table
shows TTFT unchanged at 3000 ms; in reality it **grows slightly**, which is the
TTFT-vs-ITL tradeoff real systems expose as a chunk-size knob.

*Simplification:* these chunks don't attend to earlier chunks' KV (no cache
carried), so chunked total compute slightly underestimates real chunked prefill
— the scheduling lesson is unchanged.

In [ ]:
P, C = 4096, 512
prompt     = torch.randint(0, 2048, (1, P), device=device)
decode_tok = torch.randint(0, 2048, (1, 1), device=device)   # "another user's" token

# (a) unchunked: a decode arriving mid-prefill waits for the WHOLE prefill
t_full = time_ms(lambda: model(prompt))

# (b) chunked: one iteration = one 512-token chunk + the decode token
chunks = prompt.split(C, dim=1)
t_iter = time_ms(lambda: (model(chunks[0]), model(decode_tok)))

print(f"unchunked prefill (TTFT)          : {t_full:7.2f} ms  <- max decode stall")
print(f"one chunked iteration             : {t_iter:7.2f} ms  <- max decode stall now")
print(f"chunked TTFT = {len(chunks)} iterations    : {len(chunks)*t_iter:7.2f} ms")

## 3. Mixed batches: padding waste vs real tokens

Claim: naively batching a 512-token prefill with two 1-token decodes pads to
3x512 positions -> (512+1+1)/1536 = **33% efficiency**. The mixed approach only
touches real tokens, and time scales with real token count.

*Simplification:* `flat` treats the group as one sequence; a true mixed batch
splits attention per token group — that is exactly what FlashInfer's separate
`BatchPrefillWithPagedKVCacheWrapper` / `BatchDecodeWithPagedKVCacheWrapper`
calls in the chapter do (and why they can't run on a T4).

In [ ]:
prefill = torch.randint(0, 2048, (1, 512), device=device)
dec1 = torch.randint(0, 2048, (1, 1), device=device)
dec2 = torch.randint(0, 2048, (1, 1), device=device)

# naive: pad the two 1-token decodes up to the prefill's 512 -> (3, 512)
padded = torch.cat([prefill, F.pad(dec1, (0, 511)), F.pad(dec2, (0, 511))])
# mixed: only the real tokens, one flat group -> (1, 514)
flat   = torch.cat([prefill, dec1, dec2], dim=1)

t_pad, t_mix = time_ms(lambda: model(padded)), time_ms(lambda: model(flat))
print(f"padded  (3x512 = 1536 positions): {t_pad:6.2f} ms")
print(f"real tokens (514)               : {t_mix:6.2f} ms")
print(f"efficiency 514/1536 = {514/1536:.0%},  speedup {t_pad/t_mix:.2f}x   (book: 33%)")

## 4. CUDA Graphs

Claim: ~300 launches x ~5 us = ~1.5 ms of CPU submit time is a large fraction
of a small decode step, and roughly constant — so graphs help decode (T=1) a
lot and prefill-like steps (T=512) little.

The last block demonstrates the **fixed-shape limitation** live: replaying with
a batch that wasn't captured raises `RuntimeError`. The production fix (what
vLLM does): pre-capture a ladder of shapes (1, 2, 4, 8, 16, 32...) and pad the
batch up to the next captured size, slicing the output.

On this small model decode stays memory-bound from B=1 to B=32, so the book's
batch-size trend shows up on the token-count axis instead: compare `(1, 1)`
vs `(1, 512)`.

In [ ]:
class GraphRunner:
    """Capture model(*shape) once; replay with a single launch."""
    def __init__(self, shape):                      # e.g. (1,1), (32,1), (1,512)
        self.static_in = torch.zeros(shape, dtype=torch.long, device=device)
        s = torch.cuda.Stream()                     # warmup on side stream (torch docs)
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3): model(self.static_in)
        torch.cuda.current_stream().wait_stream(s)
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.static_out = model(self.static_in)
    def __call__(self, ids):
        self.static_in.copy_(ids)                   # wrong shape -> RuntimeError
        self.graph.replay()
        return self.static_out                      # static buffer; .clone() to keep

for shape in [(1, 1), (32, 1), (1, 512)]:
    ids = torch.randint(0, 2048, shape, device=device)
    g = GraphRunner(shape)
    t_eager, t_graph = time_ms(lambda: model(ids)), time_ms(lambda: g(ids))
    print(f"{str(shape):>10}  eager {t_eager:6.2f} ms  graph {t_graph:6.2f} ms  "
          f"speedup {t_eager/t_graph:4.2f}x")

# the fixed-shape limitation, live (g is the (1,512) runner from the loop):
try:
    g(torch.randint(0, 2048, (9, 1), device=device))
except RuntimeError as e:
    print("shape not captured ->", str(e).splitlines()[0])

## 5. Overlap scheduling

Claim: sequential execution loses `scheduling / (scheduling + compute)` — the
book's example is 2/(10+2) = 17%, measured ~15%. Overlapping hides the CPU work
entirely as long as it fits inside the GPU step. `torch.cuda._sleep` gives a
deterministic GPU step independent of model timing.

Then the chapter's punchline on the real model: **graphs kill launch overhead,
overlap hides the scheduler — together the GPU never waits.**

In [ ]:
def cpu_schedule_ms(ms=2.0):                        # pure-Python scheduler work
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < ms / 1e3:
        sum(i * i for i in range(1000))

torch.cuda.synchronize(); t0 = time.perf_counter()  # calibrate the synthetic GPU step
torch.cuda._sleep(int(1e7)); torch.cuda.synchronize()
gpu_ms = (time.perf_counter() - t0) * 1e3
print(f"GPU step = {gpu_ms:.1f} ms")

ITERS = 200
# sequential: GPU idles while the CPU schedules
torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(ITERS):
    cpu_schedule_ms(2.0)
    torch.cuda._sleep(int(1e7))                     # the "forward pass"
torch.cuda.synchronize()
seq = (time.perf_counter() - t0) / ITERS * 1e3

# overlapped (book's OverlapScheduler): schedule step N+1 while step N runs
torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(ITERS):
    th = threading.Thread(target=cpu_schedule_ms, args=(2.0,))
    th.start()
    torch.cuda._sleep(int(1e7))
    th.join()
torch.cuda.synchronize()
ovl = (time.perf_counter() - t0) / ITERS * 1e3

print(f"sequential {seq:.2f} ms/iter   overlapped {ovl:.2f} ms/iter   "
      f"speedup {seq/ovl:.2f}x   (book: 17% loss, ~15% measured)")

# bonus — the chapter's punchline: overlap + CUDA graphs on the real model
g, ids = GraphRunner((8, 1)), torch.randint(0, 2048, (8, 1), device=device)

def bench(loop):
    for _ in range(20): loop()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(200): loop()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / 200 * 1e3

def seq2_step():
    cpu_schedule_ms(1.5)
    g(ids)

def ovl2_step():                                    # schedule N+1 while N replays
    th = threading.Thread(target=cpu_schedule_ms, args=(1.5,))
    th.start()
    g(ids)
    th.join()

print(f"real model: sequential {bench(seq2_step):.2f} ms/iter   "
      f"overlapped {bench(ovl2_step):.2f} ms/iter")

## 6. torch.compile: kernel fusion + automatic CUDA graphs

The chapter's manual CUDA Graph machinery is what `torch.compile` does for you.
This cell runs a 4-way decomposition on the same shapes as experiment 4:

| Path | Isolates |
|---|---|
| eager | baseline |
| eager + manual `GraphRunner` (exp 4) | pure launch elimination |
| `torch.compile(model, dynamic=False)` | Inductor kernel fusion — fewer, faster kernels, still launched one by one |
| `torch.compile(model, mode="reduce-overhead", dynamic=False)` | fusion + automatic CUDA graphs ("cudagraph trees") |

The book's limitations list, in modern form:

- automatic input copy + single replay = the manual `static_in.copy_()` /
  `graph.replay()` dance, done for you
- `dynamic=False` -> per-shape specialization = the fixed-shape limitation and
  the production shape ladder (batch sizes 1, 2, 4, 8, 16, 32)
- first call per shape pays Inductor compilation (seconds) = "Warmup cost"
- reduce-overhead outputs are overwritten by the next call = the `static_out`
  caveat from experiment 4

**Expect the first run of this cell to take a few minutes of compilation.**
One compiled wrapper is reused across all three shapes on purpose: each new
shape triggers a recompile — watch the first-call times.

In [ ]:
compiled   = torch.compile(model, dynamic=False)
compiled_ro = torch.compile(model, mode="reduce-overhead", dynamic=False)

def first_call_ms(fn):
    # wall time of one call incl. sync -- on a new shape this is the compile cost
    torch.cuda.synchronize(); t0 = time.perf_counter()
    fn(); torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1e3

for shape in [(1, 1), (32, 1), (1, 512)]:
    ids = torch.randint(0, 2048, shape, device=device)

    t_eager = time_ms(lambda: model(ids))
    g = GraphRunner(shape)                          # manual graphs, as in exp 4
    t_manual = time_ms(lambda: g(ids))

    c_first = first_call_ms(lambda: compiled(ids))
    t_comp  = time_ms(lambda: compiled(ids))
    o_first = first_call_ms(lambda: compiled_ro(ids))
    t_over  = time_ms(lambda: compiled_ro(ids))

    print(f"{str(shape):>10}  eager {t_eager:6.2f} | manual-graph {t_manual:6.2f} | "
          f"compile {t_comp:6.2f} (first call {c_first/1e3:5.1f}s) | "
          f"reduce-overhead {t_over:6.2f} (first {o_first/1e3:5.1f}s)")
    print(f"{'':>10}  speedup vs eager:  manual {t_eager/t_manual:4.2f}x  "
          f"compile {t_eager/t_comp:4.2f}x  reduce-overhead {t_eager/t_over:4.2f}x")
    # NOTE: compiled_ro outputs are invalidated by the next call --
    # .clone() anything you want to keep (same as GraphRunner's static_out).

## Notes & caveats

- Decode steps here are bare 1-token forwards (no KV-cache reads) — the kernel
  *count* matches reality, which is what drives the launch-overhead result.
- Chunks don't attend to earlier chunks' KV (no cache carried), so chunked
  total compute slightly underestimates real chunked prefill.
- torch.compile caveats (experiment 6): `mode="reduce-overhead"` returns
  tensors that the *next* call overwrites — `.clone()` anything you keep;
  `dynamic=False` makes every new shape recompile (the book's fixed-shape
  limitation in modern form); the first call per shape pays Inductor
  compilation — the book's "Warmup cost".
- All timings use CUDA events after warmup; expect run-to-run noise of a few
  percent. Rough T4 expectations: ~3x for the mixed-vs-padded batch, ~2-3x
  graph speedup at `(1,1)` (reduce-overhead at least that), ~1.3x for
  overlap — the chapter's shape of results at 1/100th the scale.